# TookaBERT, top four blocks, 10% of the reviews

Same training recipe as the full TookaBERT top-four notebook, on 10% of the reviews. Sign in with a Colab account. The notebook downloads the shared folder `1eyfGAVKe-9nVXnJaQQWWVM2_Zb9fLa4u` and writes results to that account's `MyDrive/digikala-experiments`.

The 10% is a random draw with seed 2026, so this notebook and the ParsBERT 10% notebook keep the same reviews. Counting tokens for every row is skipped.

Blocks 8–11 train at `2e-5` and the classifier trains at `1e-4`. One epoch is about a tenth of the full run, near 1.5 hours on a T4 instead of about 15. There is no second epoch. The scores belong to this slice. They do not replace the full-set validation macro F1 of 0.6722. Finished weights go to `runs/PartAI--TookaBERT-Base--top4_sample10`.


In [ ]:
MODEL_NAME = "PartAI/TookaBERT-Base"
MAX_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 16
GRAD_ACCUM = 2
BLOCK_LR = 2e-5
HEAD_LR = 1e-4
TOP_BLOCKS = 4
SEED = 2026
SAMPLE_FRACTION = 0.10

DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments"
WEIGHTS_FOLDER_ID = "1eyfGAVKe-9nVXnJaQQWWVM2_Zb9fLa4u"
START_DIR = "/content/tookabert-start"
DRIVE_FILE_ID = "1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr"
DATA_ZIP = "/content/data/digikala-comments-preprocessed.csv.zip"
LOCAL_CSV = "/content/data/digikala-comments-preprocessed.csv"

LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: code for code, label in enumerate(LABELS)}
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15


## Drive, the saved weights, and the CSV

Mount this account's Drive. That mount is where results are saved. Download the TookaBERT `best` folder from the shared link onto `/content/tookabert-start`, then download the preprocessed CSV onto the Colab disk. The notebook reads `body` and `recommendation_status` only.


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import polars as pl
import torch
from google.colab import drive
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

import gdown

drive.mount("/content/drive")
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "runs").mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "logs").mkdir(parents=True, exist_ok=True)
(Path(DRIVE_DIR) / "best").mkdir(parents=True, exist_ok=True)


def slug(model_name: str) -> str:
    return model_name.replace("/", "--")


START_DIR = Path(START_DIR)
if not (START_DIR / "model.safetensors").is_file():
    START_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(
        id=WEIGHTS_FOLDER_ID,
        output=str(START_DIR),
        quiet=False,
        remaining_ok=True,
    )
if not (START_DIR / "model.safetensors").is_file():
    nested = list(START_DIR.glob("*/model.safetensors"))
    if len(nested) == 1:
        START_DIR = nested[0].parent
if not (START_DIR / "config.json").is_file() or not (START_DIR / "model.safetensors").is_file():
    raise FileNotFoundError(
        "Could not download the shared TookaBERT weights "
        f"{WEIGHTS_FOLDER_ID} into {START_DIR}."
    )
previous_f1 = 0.0
metrics_path = START_DIR / "metrics.json"
if metrics_path.is_file():
    previous_f1 = float(json.loads(metrics_path.read_text()).get("f1") or 0.0)
print("starting weights", START_DIR)
print("results drive", DRIVE_DIR)
print("full-data validation macro F1 of these weights", round(previous_f1, 4))

zip_path = Path(DATA_ZIP)
zip_path.parent.mkdir(parents=True, exist_ok=True)
if not zip_path.is_file():
    gdown.download(id=DRIVE_FILE_ID, output=str(zip_path), quiet=False)
if not zip_path.is_file():
    raise FileNotFoundError(f"Could not download Drive file {DRIVE_FILE_ID} to {zip_path}.")
with zipfile.ZipFile(zip_path) as archive:
    member = "digikala-comments-preprocessed.csv"
    names = [
        name
        for name in archive.namelist()
        if name.endswith(".csv") and not name.startswith("__MACOSX")
    ]
    if member not in names:
        if len(names) != 1:
            raise FileNotFoundError(f"Expected one CSV in the zip, found {names}")
        member = names[0]
    info = archive.getinfo(member)
    local_csv = Path(LOCAL_CSV)
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    if (not local_csv.is_file()) or local_csv.stat().st_size != info.file_size:
        print("extracting", member, info.file_size, "bytes")
        with archive.open(info) as source, local_csv.open("wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("csv:", local_csv, "bytes", local_csv.stat().st_size)


## A random 10% slice, then the same split

Draw 10% of the labeled reviews at random with seed 2026. The same seed keeps the same reviews in both notebooks. The other 90% is dropped. The kept rows are then shuffled by class with seed 2026 and cut 70/15/15. Expect about 368,000 train rows, 79,000 validation, and 79,000 test.


In [ ]:
reviews = pl.read_csv(
    LOCAL_CSV,
    columns=["body", "recommendation_status"],
    schema_overrides={"body": pl.String, "recommendation_status": pl.String},
)
labeled = reviews.filter(
    pl.col("body").is_not_null()
    & (pl.col("body").str.strip_chars() != "")
    & pl.col("recommendation_status").is_in(LABELS)
).with_columns(
    pl.col("recommendation_status").replace_strict(LABEL_CODE).alias("label_id")
)


full_rows = labeled.height
n_keep = int(round(full_rows * SAMPLE_FRACTION))
labeled = labeled.sample(n=n_keep, shuffle=True, seed=SPLIT_SEED)
print("kept", labeled.height, "of", full_rows, "fraction", round(labeled.height / full_rows, 4))

parts = []
for label in LABELS:
    group = labeled.filter(pl.col("recommendation_status") == label)
    shuffled = group.sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    count = shuffled.height
    n_train = int(count * TRAIN_SHARE)
    n_validation = int(count * VALIDATION_SHARE)
    n_test = count - n_train - n_validation
    parts.append(
        shuffled.with_columns(
            pl.Series(
                "split",
                (["train"] * n_train)
                + (["validation"] * n_validation)
                + (["test"] * n_test),
            )
        )
    )
labeled = pl.concat(parts)
train_frame = labeled.filter(pl.col("split") == "train")
valid_frame = labeled.filter(pl.col("split") == "validation")
test_frame = labeled.filter(pl.col("split") == "test")
print("train", train_frame.height, "validation", valid_frame.height, "test", test_frame.height)
print(
    labeled.group_by("split", "recommendation_status")
    .len()
    .sort("split", "recommendation_status")
)


## Top four blocks

Blocks 8, 9, 10, and 11 train at `2e-5`. The classifier trains at `1e-4`. The bottom eight blocks stay frozen. The loss is class-weighted cross-entropy, and the printed precision, recall, and F1 are macro averages.


In [ ]:
class ReviewSet(Dataset):
    def __init__(self, frame, tokenizer, max_length: int):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("label_id").to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index] or "",
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = int(self.labels[index])
        return encoded


def metric_bundle(actual, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average="macro", zero_division=0
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        actual, predicted, labels=[0, 1, 2], average=None, zero_division=0
    )
    report = {
        "accuracy": float(accuracy_score(actual, predicted)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    for index, name in enumerate(LABELS):
        report[f"{name}_precision"] = float(per_precision[index])
        report[f"{name}_recall"] = float(per_recall[index])
        report[f"{name}_f1"] = float(per_f1[index])
        report[f"{name}_support"] = int(per_support[index])
    return report


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predicted = np.argmax(logits, axis=-1)
    scores = metric_bundle(labels, predicted)
    return {key: scores[key] for key in ("accuracy", "precision", "recall", "f1")}


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(outputs.logits.device)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


class EpochClock(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        import time
        self.started = time.perf_counter()

    def on_log(self, args, state, control, logs=None, **kwargs):
        import time
        if state.global_step != 100 or not state.max_steps or not getattr(self, "started", None):
            return
        hours = (time.perf_counter() - self.started) * state.max_steps / 100 / 3600
        print(f"step 100 of {state.max_steps}; this epoch is about {hours:.1f} hours")


def unfreeze_top_blocks(model, n_blocks: int) -> None:
    prefix = "bert"
    for name, param in model.named_parameters():
        param.requires_grad = not name.startswith(prefix + ".")
    layers = model.bert.encoder.layer
    if n_blocks > len(layers):
        raise ValueError(f"TookaBERT has {len(layers)} blocks, requested {n_blocks}.")
    for layer in layers[-n_blocks:]:
        for param in layer.parameters():
            param.requires_grad = True
    for name, param in model.named_parameters():
        if "classifier" in name:
            param.requires_grad = True
    first = len(layers) - n_blocks
    print("training blocks", list(range(first, len(layers))), "and the classifier")


def parameter_groups(model, block_lr: float, head_lr: float):
    head, block = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("bert."):
            block.append(param)
        else:
            head.append(param)
    return [
        {"params": block, "lr": block_lr},
        {"params": head, "lr": head_lr},
    ]


def training_arguments(output_dir: str) -> TrainingArguments:
    shared = dict(
        output_dir=output_dir,
        learning_rate=BLOCK_LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
    )
    try:
        return TrainingArguments(eval_strategy="no", **shared)
    except TypeError:
        return TrainingArguments(evaluation_strategy="no", **shared)


def trainer_token_kwargs(tokenizer):
    import inspect
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        return {"processing_class": tokenizer}
    return {"tokenizer": tokenizer}


## Train one epoch, then score the 10% slice

Trainer checkpoints stay on the Colab disk. After the epoch the cell copies the model files to `runs/{slug}--top4_sample10` before scoring. Validation and test are the 10% slice, so the scores are not a new measurement of the full dataset. A copy is written to `best/{slug}--top4_sample10`.


In [ ]:
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(START_DIR)
model = AutoModelForSequenceClassification.from_pretrained(START_DIR)
unfreeze_top_blocks(model, TOP_BLOCKS)
trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
print("trainable parameters", trainable)

class_weights = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.array([0, 1, 2]),
        y=train_frame.get_column("label_id").to_numpy(),
    ),
    dtype=torch.float,
)
train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
test_set = ReviewSet(test_frame, tokenizer, MAX_LENGTH)
collator = DataCollatorWithPadding(tokenizer)
token_kwargs = trainer_token_kwargs(tokenizer)

stage = "top4_sample10"
local_run = Path("/content/tookabert-top4")
drive_run = Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"
local_run.mkdir(parents=True, exist_ok=True)
drive_run.mkdir(parents=True, exist_ok=True)
groups = parameter_groups(model, BLOCK_LR, HEAD_LR)


def publish(metrics: dict) -> None:
    model.save_pretrained(drive_run)
    tokenizer.save_pretrained(drive_run)
    (drive_run / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print("copied weights to", drive_run)


def make_trainer():
    trainer = WeightedTrainer(
        model=model,
        args=training_arguments(str(local_run)),
        train_dataset=train_set,
        data_collator=collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[EpochClock()],
        **token_kwargs,
    )

    def create_optimizer():
        trainer.optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
        return trainer.optimizer

    trainer.create_optimizer = create_optimizer
    return trainer


def score(frame_set):
    holder = Trainer(
        model=model,
        args=training_arguments("/tmp/tookabert-score"),
        data_collator=collator,
        **token_kwargs,
    )
    prediction = holder.predict(frame_set)
    return metric_bundle(prediction.label_ids, prediction.predictions.argmax(axis=-1))


def save_weights(metrics: dict) -> None:
    publish(metrics)


print("epoch 1")
make_trainer().train()
model.save_pretrained(drive_run)
tokenizer.save_pretrained(drive_run)
print("copied weights to the new Drive before scoring", drive_run)

validation_metrics = score(valid_set)
validation_metrics.update({"split": "validation", "stage": stage, "model_name": MODEL_NAME, "epoch": 1})
save_weights(validation_metrics)
print("validation epoch 1", {key: validation_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})

kept = validation_metrics
print("one epoch on the 10% slice")

test_metrics = score(test_set)
test_metrics.update({"split": "test", "stage": stage, "model_name": MODEL_NAME})
(drive_run / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print("test", {key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})

log_path = Path(DRIVE_DIR) / "logs" / f"{slug(MODEL_NAME)}--{stage}.jsonl"
log_path.parent.mkdir(parents=True, exist_ok=True)
with log_path.open("a") as handle:
    handle.write(json.dumps(kept) + "\n")
    handle.write(json.dumps(test_metrics) + "\n")

best_dir = Path(DRIVE_DIR) / "best" / f"{slug(MODEL_NAME)}--{stage}"
best_dir.mkdir(parents=True, exist_ok=True)
for item in drive_run.iterdir():
    if item.is_file():
        (best_dir / item.name).write_bytes(item.read_bytes())
print("saved this 10% run", best_dir, "macro F1", round(kept["f1"], 4))


## More epochs from the model already in memory

Run this after the epoch-1 cell has finished. Do not run it while that cell is still going. The runtime is still connected, and `model` is the epoch-1 weights. Each pass below trains one more epoch on that same object. AdamW starts fresh for each pass. Nothing above this cell is read again.

`EXTRA_EPOCHS = 2` means epochs 2 and 3. Change that number before running if you want a different count. After each epoch the cell scores the 10% validation slice. Drive is updated only when the macro F1 is at least as high as the best epoch so far. The test split is scored once, on that best epoch.


In [ ]:
EXTRA_EPOCHS = 2


def weight_snapshot():
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def restore_snapshot(snapshot) -> None:
    model.load_state_dict(snapshot)


if "kept" not in globals():
    raise RuntimeError("The epoch-1 cell has not finished scoring yet. Wait for it, then run this cell.")

best_epoch = int(kept.get("epoch", 1))
best_f1 = float(kept["f1"])
best_snapshot = weight_snapshot()
print("continuing from the in-memory model, epoch", best_epoch, "validation macro F1", round(best_f1, 4))

for offset in range(1, EXTRA_EPOCHS + 1):
    epoch = best_epoch + offset
    print("epoch", epoch)
    make_trainer().train()
    metrics = score(valid_set)
    metrics.update({"split": "validation", "stage": stage, "model_name": MODEL_NAME, "epoch": epoch})
    print(
        "validation epoch",
        epoch,
        {key: metrics[key] for key in ("accuracy", "precision", "recall", "f1")},
    )
    log_path = Path(DRIVE_DIR) / "logs" / f"{slug(MODEL_NAME)}--{stage}.jsonl"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a") as handle:
        handle.write(json.dumps(metrics) + "\n")
    if metrics["f1"] >= best_f1:
        best_f1 = float(metrics["f1"])
        best_epoch = epoch
        best_snapshot = weight_snapshot()
        kept = metrics
        save_weights(metrics)
        print("copied epoch", epoch, "to", drive_run)
    else:
        print("epoch", epoch, "did not beat epoch", best_epoch, "; Drive still holds that epoch")

restore_snapshot(best_snapshot)
print("test will use epoch", best_epoch)
test_metrics = score(test_set)
test_metrics.update({"split": "test", "stage": stage, "model_name": MODEL_NAME, "epoch": best_epoch})
(drive_run / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print("test", {key: test_metrics[key] for key in ("accuracy", "precision", "recall", "f1")})
with log_path.open("a") as handle:
    handle.write(json.dumps(test_metrics) + "\n")

best_dir = Path(DRIVE_DIR) / "best" / f"{slug(MODEL_NAME)}--{stage}"
best_dir.mkdir(parents=True, exist_ok=True)
for item in drive_run.iterdir():
    if item.is_file():
        (best_dir / item.name).write_bytes(item.read_bytes())
print("saved the best of these epochs", best_dir, "epoch", best_epoch, "macro F1", round(best_f1, 4))
